In [4]:
import numpy as np
import scipy.sparse as sp
import os
from itertools import product
from typing import Dict, Tuple, Callable, List
from joblib import Parallel, delayed
import time

# ============================================================
# 0) Pauli algebra (sparse) + apply to statevector
# ============================================================

EPS = 1e-14

_PM = {
    ('I','I'):(1,'I'), ('I','X'):(1,'X'), ('I','Y'):(1,'Y'), ('I','Z'):(1,'Z'),
    ('X','I'):(1,'X'), ('Y','I'):(1,'Y'), ('Z','I'):(1,'Z'),
    ('X','X'):(1,'I'), ('Y','Y'):(1,'I'), ('Z','Z'):(1,'I'),
    ('X','Y'):(1j,'Z'), ('Y','X'):(-1j,'Z'),
    ('Y','Z'):(1j,'X'), ('Z','Y'):(-1j,'X'),
    ('Z','X'):(1j,'Y'), ('X','Z'):(-1j,'Y'),
}

PauliKey = Tuple[Tuple[int, str], ...]
PauliSum = Dict[PauliKey, complex]

def _clean(paulisum: dict) -> dict:
    return {Pk: c for Pk, c in paulisum.items() if abs(c) > EPS}

def multiply_pauli_strings(P, Q):
    d = dict(P)
    ph = 1.0 + 0j
    for q, p2 in Q:
        p1 = d.get(q, 'I')
        a, p3 = _PM[(p1, p2)]
        ph *= a
        if p3 == 'I':
            d.pop(q, None)
        else:
            d[q] = p3
    return ph, tuple(sorted(d.items()))

def multiply_pauli_sums(A, B):
    out = {}
    for P, a in A.items():
        for Q, b in B.items():
            ph, R = multiply_pauli_strings(P, Q)
            out[R] = out.get(R, 0.0 + 0j) + a * b * ph
    return _clean(out)

def add_pauli_sums(A, B):
    out = dict(A)
    for P, c in B.items():
        out[P] = out.get(P, 0.0 + 0j) + c
    return _clean(out)

def scale_pauli_sum(A, s):
    if abs(s) < EPS or not A:
        return {}
    return _clean({P: s * c for P, c in A.items()})

def paulisum_to_single_term(poly):
    if len(poly) != 1:
        raise ValueError(f"Expected single-term PauliSum, got {len(poly)} terms.")
    (Pk, c), = poly.items()
    return Pk, c

def _parity_uint64(x):
    x = x.copy()
    x ^= x >> np.uint64(32)
    x ^= x >> np.uint64(16)
    x ^= x >> np.uint64(8)
    x ^= x >> np.uint64(4)
    x ^= x >> np.uint64(2)
    x ^= x >> np.uint64(1)
    return (x & np.uint64(1)).astype(np.uint8)

def apply_pauli_string(psi, P):
    flip = np.uint64(0)
    sign = np.uint64(0)
    ny = 0
    for q, p in P:
        bit = np.uint64(1) << np.uint64(q)
        if p == 'X':
            flip ^= bit
        elif p == 'Z':
            sign ^= bit
        elif p == 'Y':
            flip ^= bit
            sign ^= bit
            ny += 1

    idx = np.arange(psi.size, dtype=np.uint64)
    idx2 = idx ^ flip

    v = idx & sign
    parity = _parity_uint64(v)
    signs = 1 - 2 * parity.astype(np.int8)

    phase = ((-1j) ** ny) * signs.astype(np.complex128)
    return phase * psi[idx2]

def apply_paulisum_to_statevector(psi: np.ndarray, poly: dict) -> np.ndarray:
    out = np.zeros_like(psi, dtype=np.complex128)
    for Pk, c in poly.items():
        out += c * apply_pauli_string(psi, Pk)
    return out

def expval_pauli_sum(psi, poly):
    total = 0.0 + 0j
    for Pk, c in poly.items():
        total += c * np.vdot(psi, apply_pauli_string(psi, Pk))
    return total

# ============================================================
# 1) JW gamma map
# ============================================================

def jw_gamma_to_pauli(n_modes):
    def gamma_to_pauli(a):
        if a < 0 or a >= 2 * n_modes:
            raise IndexError("Majorana index out of range.")
        j = a // 2
        base = 'X' if (a % 2 == 0) else 'Y'
        ops = {k: 'Z' for k in range(j)}
        ops[j] = base
        return {tuple(sorted(ops.items())): 1.0 + 0j}
    return gamma_to_pauli

# ============================================================
# 2) TT path-operator Majorana map
# ============================================================

def ternary_tree_majorana_map(n_modes: int):
    need = 2 * n_modes
    h = 1
    while 3**h < need:
        h += 1

    sigma = {0: 'X', 1: 'Y', 2: 'Z'}

    def node_on_path(depth: int, p_digits) -> int:
        v = 0
        for d in range(depth):
            v = 3 * v + 1 + p_digits[d]
        return v

    def path_pauli_operator(p_digits):
        ops = {}
        for depth, dig in enumerate(p_digits):
            q = node_on_path(depth, p_digits)
            ops[q] = sigma[dig]
        return tuple(sorted(ops.items()))

    all_paths = list(product([0, 1, 2], repeat=h))
    chosen = all_paths[:need]
    pauli_keys = [path_pauli_operator(p) for p in chosen]

    Q = max(q for Pk in pauli_keys for (q, _) in Pk) if pauli_keys else -1
    n_qubits = Q + 1

    def gamma_to_pauli(a: int):
        if a < 0 or a >= 2 * n_modes:
            raise IndexError("Majorana index out of range.")
        return {pauli_keys[a]: 1.0 + 0j}

    return gamma_to_pauli, n_qubits

def tt_c_paulisum(gamma_tt, j: int) -> dict:
    return add_pauli_sums(
        scale_pauli_sum(gamma_tt(2*j), 0.5),
        scale_pauli_sum(gamma_tt(2*j + 1), +0.5j),
    )

def tt_cdagger_paulisum(gamma_tt, j: int) -> dict:
    return add_pauli_sums(
        scale_pauli_sum(gamma_tt(2*j), 0.5),
        scale_pauli_sum(gamma_tt(2*j + 1), -0.5j),
    )

def project_onto_pauli_eigenspace(psi, Pk, eigenvalue: int):
    if eigenvalue not in (+1, -1):
        raise ValueError("eigenvalue must be +1 or -1.")
    psi2 = 0.5 * (psi + eigenvalue * apply_pauli_string(psi, Pk))
    nrm = np.linalg.norm(psi2)
    if nrm < EPS:
        raise RuntimeError("Projection annihilated the state.")
    return psi2 / nrm

def build_tt_fermionic_vacuum_state(
    gamma_tt, n_modes: int, n_qubits_tt: int,
    check_commutation: bool = True, verbose: bool = True
):
    psi = np.zeros(1 << n_qubits_tt, dtype=np.complex128)
    psi[0] = 1.0 + 0j

    bil_Pk, bil_eig = [], []
    for j in range(n_modes):
        Bj = multiply_pauli_sums(gamma_tt(2*j), gamma_tt(2*j + 1))
        Pk, c = paulisum_to_single_term(Bj)

        e = (-1.0) / (1j * c)
        if not (abs(e - 1) < 1e-8 or abs(e + 1) < 1e-8):
            raise RuntimeError(f"Mode j={j}: need e={e} not ±1 (c={c}).")

        bil_Pk.append(Pk)
        bil_eig.append(+1 if abs(e - 1) < 1e-8 else -1)

    for j in range(n_modes):
        psi = project_onto_pauli_eigenspace(psi, bil_Pk[j], bil_eig[j])

    return psi

# ============================================================
# 3) Matrix backend: PauliSum -> SCIPY SPARSE MATRIX
# ============================================================

_I2 = sp.eye(2, format='csr', dtype=complex)
_X  = sp.csr_matrix([[0, 1], [1, 0]], dtype=complex)
_Y  = sp.csr_matrix([[0, -1j], [1j, 0]], dtype=complex)
_Z  = sp.csr_matrix([[1, 0], [0, -1]], dtype=complex)
_PMAT_SP = {'I': _I2, 'X': _X, 'Y': _Y, 'Z': _Z}

def pauli_key_to_matrix(Pk, n_qubits: int):
    ops = ['I'] * n_qubits
    for q, p in Pk:
        ops[q] = p
    M = sp.csr_matrix([[1]], dtype=complex)
    for q in range(n_qubits - 1, -1, -1):
        M = sp.kron(M, _PMAT_SP[ops[q]], format='csr')
    return M

def paulisum_to_matrix(poly, n_qubits: int):
    dim = 1 << n_qubits
    H = sp.csr_matrix((dim, dim), dtype=complex)
    for Pk, c in poly.items():
        H += c * pauli_key_to_matrix(Pk, n_qubits)
    return H

# ============================================================
# 5) Heisenberg correlations at t=0:
# ============================================================

def correlation_matrix_majoranas_heisenberg(psi, gamma_map, n_modes: int, n_qubits: int):
    twoN = 2 * n_modes
    G = []
    for a in range(twoN):
        Pk, c = paulisum_to_single_term(gamma_map(a))
        G.append(c * pauli_key_to_matrix(Pk, n_qubits))
    C = np.zeros((twoN, twoN), dtype=np.complex128)
    for a in range(twoN):
        for b in range(twoN):
            C[a, b] = 1j * np.vdot(psi, G[a] @ (G[b] @ psi))
    return C

# ============================================================
# MINIMAL FIX: build B = gamma_a(t) gamma_b(t) as wedge (c<d)
# ============================================================

def build_majorana_bilinear_from_rows(G_list, ra, rb, include_identity=True):
    twoN = len(ra)
    dim = G_list[0].shape[0]
    B = sp.csr_matrix((dim, dim), dtype=np.complex128)

    for c in range(twoN):
        Gc = G_list[c]
        for d in range(c + 1, twoN):
            w = ra[c] * rb[d] - ra[d] * rb[c]
            if abs(w) < EPS:
                continue
            B += w * (Gc @ G_list[d])

    if include_identity:
        dot = np.dot(ra, rb)
        if abs(dot) > 1e-12:
            B += dot * sp.eye(dim, dtype=np.complex128, format='csr')
    return B

def conj_exp_majorana_bilinear_matrix_quadratic(
    gamma_map, n_modes: int, n_qubits: int, K: np.ndarray,
    t: float, theta: float, a: int, b: int
):
    twoN = 2 * n_modes
    w, V = np.linalg.eigh(-1j * K * t)
    d = np.exp(1j * w)
    Rm = (V * d[None, :]) @ V.conj().T
    Rm = np.real_if_close(Rm, tol=1e-10)

    ra = np.asarray(Rm[a, :], dtype=np.complex128)
    rb = np.asarray(Rm[b, :], dtype=np.complex128)

    G = []
    for c in range(twoN):
        Pk, coeff = paulisum_to_single_term(gamma_map(c))
        G.append(coeff * pauli_key_to_matrix(Pk, n_qubits))

    dim = 1 << n_qubits
    I = sp.eye(dim, dtype=np.complex128, format='csr')

    B = build_majorana_bilinear_from_rows(G, ra, rb, include_identity=True)

    return (np.cos(theta) * I) + (np.sin(theta) * B)

# ============================================================
# Number-conserving h -> Majorana K
# ============================================================

def h_to_majorana_K(h: np.ndarray) -> np.ndarray:
    h = np.asarray(h, dtype=np.complex128)
    n = h.shape[0]
    X = np.real(h)
    Y = np.imag(h)
    K = np.zeros((2*n, 2*n), dtype=np.float64)
    K[0::2, 0::2] =  2.0 * Y
    K[1::2, 1::2] =  2.0 * Y
    K[0::2, 1::2] = -2.0 * X
    K[1::2, 0::2] =  2.0 * X
    return 0.5 * (K - K.T)

def h_eq14_1d_nn(N: int, Vnn: float, delta: float, mu: float, periodic: bool = False):
    h = np.zeros((N, N), dtype=np.complex128)
    np.fill_diagonal(h, -mu)
    def sgn(i: int) -> float: return 1.0 if (i % 2 == 0) else -1.0
    last = N if periodic else N - 1
    for i in range(last):
        j = (i + 1) % N
        t_i = Vnn * (1.0 + sgn(i) * (delta / 2.0))
        h[i, j] += -t_i
        h[j, i] += -t_i
    return h

def build_single_particle_uniform_state(psi_vac, gamma_map, n_modes):
    psi_sum = np.zeros_like(psi_vac, dtype=np.complex128)
    for j in range(n_modes):
        cdag = add_pauli_sums(
            scale_pauli_sum(gamma_map(2*j), 0.5),
            scale_pauli_sum(gamma_map(2*j + 1), -0.5j),
        )
        psi_sum += apply_paulisum_to_statevector(psi_vac, cdag)
    psi_sum /= np.sqrt(n_modes)
    nrm = np.linalg.norm(psi_sum)
    if abs(nrm - 1.0) > 1e-12: psi_sum /= nrm
    return psi_sum

def select_mu_for_uniform_manybody_ground_state(n_modes: int, t: float):
    N = n_modes
    lower = -2.0 * t
    upper = -2.0 * t * np.cos(2.0 * np.pi / N)
    return 0.5 * (lower + upper)

def build_quadratic_hamiltonian_matrix(h: np.ndarray, encoding: str):
    h = np.asarray(h, dtype=np.complex128)
    n = h.shape[0]
    if encoding == "jw":
        gamma = jw_gamma_to_pauli(n)
        n_qubits = n
    elif encoding == "tt":
        gamma, n_qubits = ternary_tree_majorana_map(n)
    else:
        raise ValueError("encoding must be 'jw' or 'tt'")

    H = {}
    for i in range(n):
        for j in range(n):
            if abs(h[i, j]) < EPS: continue
            cdag_i = add_pauli_sums(scale_pauli_sum(gamma(2*i), 0.5), scale_pauli_sum(gamma(2*i+1), -0.5j))
            c_j = add_pauli_sums(scale_pauli_sum(gamma(2*j), 0.5), scale_pauli_sum(gamma(2*j+1), +0.5j))
            term = multiply_pauli_sums(cdag_i, c_j)
            H = add_pauli_sums(H, scale_pauli_sum(term, h[i, j]))
    return paulisum_to_matrix(H, n_qubits)

# ============================================================
# Classical shadows from a statevector
# ============================================================

def _sample_pauli_bases(shots: int, n_qubits: int) -> np.ndarray:
    return np.random.randint(0, 3, size=(shots, n_qubits), dtype=np.int8)

def _apply_single_qubit_U_to_state(psi: np.ndarray, n_qubits: int, q: int, U: np.ndarray) -> np.ndarray:
    # Reshape statevector to a tensor where each axis represents a qubit
    psi_tensor = psi.reshape([2] * n_qubits)
    
    # In your convention, q=0 maps to the last axis in C-contiguous order
    axis = n_qubits - 1 - q
    
    # Contract the unitary with the target qubit axis
    out_tensor = np.tensordot(U, psi_tensor, axes=([1], [axis]))
    
    # np.tensordot moves the operated axis to the front (axis 0), so we must move it back
    out_tensor = np.moveaxis(out_tensor, 0, axis)
    
    # Flatten back to a statevector
    return out_tensor.reshape(-1)

def _rotate_to_Z_basis(psi: np.ndarray, bases_row: np.ndarray, n_qubits: int) -> np.ndarray:
    H = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=np.complex128)
    Sdg = np.array([[1, 0], [0, -1j]], dtype=np.complex128)
    out = psi
    for q in range(n_qubits):
        b = int(bases_row[q])
        if b == 0: out = _apply_single_qubit_U_to_state(out, n_qubits, q, H)
        elif b == 1:
            out = _apply_single_qubit_U_to_state(out, n_qubits, q, Sdg)
            out = _apply_single_qubit_U_to_state(out, n_qubits, q, H)
    return out

def _sample_bitstring_from_probs(probs: np.ndarray) -> int:
    cdf = np.cumsum(probs)
    r = np.random.random()
    return int(np.searchsorted(cdf, r, side="right"))

def _simulate_single_shot(psi: np.ndarray, basis_row: np.ndarray, n_qubits: int) -> List[int]:
    psi_rot = _rotate_to_Z_basis(psi, basis_row, n_qubits)
    probs = (psi_rot.conj() * psi_rot).real
    probs = np.maximum(probs, 0.0)
    probs /= probs.sum()
    outcome = _sample_bitstring_from_probs(probs)
    return [(outcome >> q) & 1 for q in range(n_qubits)]

def sample_classical_shadows_statevector(
    psi: np.ndarray, 
    shots: int, 
    max_cpus: int = None
) -> Tuple[np.ndarray, np.ndarray]:
    
    if max_cpus is None:
        max_cpus = os.cpu_count() or 1
        
    psi = np.asarray(psi, dtype=np.complex128).reshape(-1)
    n_qubits = int(np.round(np.log2(psi.size)))
    norm = np.linalg.norm(psi)
    psi = psi / norm

    bases = _sample_pauli_bases(shots, n_qubits)

    results = Parallel(n_jobs=max_cpus)(
        delayed(_simulate_single_shot)(psi, bases[s], n_qubits) 
        for s in range(shots)
    )

    bits = np.array(results, dtype=np.uint8)

    return bases, bits

def pauli_key_to_dense_list(Pk: PauliKey, n_qubits: int) -> List[str]:
    out = ['I'] * n_qubits
    for q, p in Pk: out[q] = p
    return out

def dense_list_to_pauli_key(ps: List[str]) -> PauliKey:
    return tuple((q, ps[q]) for q in range(len(ps)) if ps[q] != 'I')

def multiply_pauli_keys(P: PauliKey, Q: PauliKey, n_qubits: int) -> Tuple[PauliKey, complex]:
    p_list = pauli_key_to_dense_list(P, n_qubits)
    q_list = pauli_key_to_dense_list(Q, n_qubits)
    out = ['I'] * n_qubits
    phase = 1.0 + 0.0j
    for i in range(n_qubits):
        ph, r = _PM[(p_list[i], q_list[i])]
        phase *= ph
        out[i] = r
    return dense_list_to_pauli_key(out), phase

def _bases_to_int(bases: np.ndarray) -> np.ndarray:
    if bases.dtype.kind in ('i', 'u'): return bases
    m = {'X':0, 'Y':1, 'Z':2}
    b = np.empty_like(bases, dtype=np.int8)
    for k, v in m.items(): b[bases == k] = v
    return b

def estimate_pauli_string_from_shadows(Pk: PauliKey, bases: np.ndarray, bits: np.ndarray, n_qubits: int) -> float:
    if len(Pk) == 0: return 1.0
    bases_i = _bases_to_int(bases)
    s = 1.0 - 2.0 * bits.astype(np.float64)
    supp = np.array([q for (q, _) in Pk], dtype=np.int64)
    pchars = [p for (_, p) in Pk]
    p_int = np.array([{'X':0,'Y':1,'Z':2}[p] for p in pchars], dtype=np.int8)
    match = np.all(bases_i[:, supp] == p_int[None, :], axis=1)
    prod = np.prod(s[:, supp], axis=1)
    w = len(supp)
    est = (3.0 ** w) * np.mean(match.astype(np.float64) * prod)
    return float(est)

def estimate_pauli_string_from_shadows_mom(
    Pk: tuple, bases: np.ndarray, bits: np.ndarray, n_qubits: int, num_blocks: int = 10
) -> float:
    if len(Pk) == 0: 
        return 1.0
        
    bases_i = _bases_to_int(bases)
    s = 1.0 - 2.0 * bits.astype(np.float64)
    supp = np.array([q for (q, _) in Pk], dtype=np.int64)
    pchars = [p for (_, p) in Pk]
    p_int = np.array([{'X':0,'Y':1,'Z':2}[p] for p in pchars], dtype=np.int8)
    
    match = np.all(bases_i[:, supp] == p_int[None, :], axis=1)
    prod = np.prod(s[:, supp], axis=1)
    w = len(supp)
    
    # 1. Calculate the individual snapshot estimator for every single shot
    snapshots = (3.0 ** w) * match.astype(np.float64) * prod
    
    # 2. Safety check: if we have fewer shots than blocks, fall back to standard mean
    actual_blocks = min(num_blocks, len(snapshots))
    if actual_blocks <= 1:
        return float(np.mean(snapshots))
        
    # 3. Median of Means: Split into blocks, compute mean of each, then take the median
    blocks = np.array_split(snapshots, actual_blocks)
    block_means = [np.mean(block) for block in blocks]
    est = np.median(block_means)
    
    return float(est)

def correlation_matrix_majoranas_from_shadows(
    bases: np.ndarray, bits: np.ndarray, gamma_map: Callable[[int], PauliSum],
    n_modes: int, n_qubits: int,
) -> np.ndarray:
    twoN = 2 * n_modes
    C = np.zeros((twoN, twoN), dtype=np.complex128)
    g_key, g_coeff = [], []
    for a in range(twoN):
        Pka, ca = paulisum_to_single_term(gamma_map(a))
        g_key.append(Pka)
        g_coeff.append(ca)

    for a in range(twoN):
        for b in range(twoN):
            Pab, phase = multiply_pauli_keys(g_key[a], g_key[b], n_qubits)
            coeff = 1j * g_coeff[a] * g_coeff[b] * phase
            expP = estimate_pauli_string_from_shadows(Pab, bases, bits, n_qubits)
            C[a, b] = coeff * expP
    return C

def correlation_matrix_majoranas_from_statevector_via_shadows(
    psi_TT: np.ndarray, gamma_map: Callable[[int], PauliSum], n_modes: int,
    n_qubits: int, shots: int,
) -> np.ndarray:
    bases, bits = sample_classical_shadows_statevector(psi_TT, shots=shots)
    return correlation_matrix_majoranas_from_shadows(bases, bits, gamma_map, n_modes, n_qubits)

def estimate_fermionic_1body_observable(C_majorana: np.ndarray, n_modes: int):
    rho = np.zeros((n_modes, n_modes), dtype=np.complex128)
    for i in range(n_modes):
        for j in range(n_modes):
            a, b = 2*i, 2*j
            ap1, bp1 = 2*i + 1, 2*j + 1
            term1 = -1j * C_majorana[a, b]
            term2 = C_majorana[a, bp1]
            term3 = -C_majorana[ap1, b]
            term4 = -1j * C_majorana[ap1, bp1]
            val = 0.25 * (term1 + term2 + term3 + term4)
            if i == j:
                rho[i, j] = 0.5 * (1.0 - C_majorana[a, ap1])
            else:
                rho[i, j] = val
    return rho


def estimate_fermionic_commutator_observable(C_combo_majorana: np.ndarray, n_modes: int):
    rho_comm = np.zeros((n_modes, n_modes), dtype=np.complex128)
    for i in range(n_modes):
        for j in range(n_modes):
            a, b = 2*i, 2*j
            ap1, bp1 = 2*i + 1, 2*j + 1
            
            # The standard mapping for the off-diagonal elements remains unchanged
            term1 = -1j * C_combo_majorana[a, b]
            term2 = C_combo_majorana[a, bp1]
            term3 = -C_combo_majorana[ap1, b]
            term4 = -1j * C_combo_majorana[ap1, bp1]
            val = 0.25 * (term1 + term2 + term3 + term4)
            
            if i == j:
                # The 0.5 constant is removed here. 
                # We only keep the Majorana expectation mapping.
                rho_comm[i, j] = -0.5 * C_combo_majorana[a, ap1]
            else:
                rho_comm[i, j] = val
                
    return rho_comm
    
# ============================================================
# 8) Build B(t) consistent with the theta-loop
# ============================================================

def build_Bt_majorana_bilinear_matrix_quadratic(
    gamma_map, n_modes: int, n_qubits: int, K: np.ndarray,
    t: float, a: int, b: int
) -> sp.spmatrix:
    twoN = 2 * n_modes
    w, V = np.linalg.eigh(-1j * K * t)
    d = np.exp(1j * w)
    Rm = (V * d[None, :]) @ V.conj().T
    Rm = np.real_if_close(Rm, tol=1e-10)

    ra = np.asarray(Rm[a, :], dtype=np.complex128)
    rb = np.asarray(Rm[b, :], dtype=np.complex128)

    G = []
    for c in range(twoN):
        Pk, coeff = paulisum_to_single_term(gamma_map(c))
        G.append(coeff * pauli_key_to_matrix(Pk, n_qubits))

    return build_majorana_bilinear_from_rows(G, ra, rb, include_identity=True)

# ============================================================
# 9) THE KEY FIX: build A_ij exactly as YOUR rho-extractor does
# ============================================================

def build_Aij_matrix_consistent_with_estimator(
    G_list: List[sp.spmatrix], n_modes: int, i: int, j: int
) -> sp.spmatrix:
    dim = G_list[0].shape[0]
    if i == j:
        I = sp.eye(dim, dtype=np.complex128, format='csr')
        ge = G_list[2*i]
        go = G_list[2*i+1]
        return 0.5 * I - 0.5 * (1j * (ge @ go))
    else:
        pairs  = [(2*i, 2*j), (2*i, 2*j+1), (2*i+1, 2*j), (2*i+1, 2*j+1)]
        coeffs = [0.25, 0.25j, -0.25j, 0.25]
        A = sp.csr_matrix((dim, dim), dtype=np.complex128)
        for (x, y), cc in zip(pairs, coeffs):
            A += cc * (G_list[x] @ G_list[y])
        return A

def compute_commutator_Rho_consistent(
    psi: np.ndarray, B_t: sp.spmatrix, G_list: List[sp.spmatrix], n_modes: int
) -> np.ndarray:
    out = np.zeros((n_modes, n_modes), dtype=np.complex128)
    for i in range(n_modes):
        for j in range(n_modes):
            Aij = build_Aij_matrix_consistent_with_estimator(G_list, n_modes, i, j)
            comm = Aij @ B_t - B_t @ Aij
            out[i, j] = (-1j) * np.vdot(psi, comm @ psi)
    return out

def combine_majorana_pair_objects_to_fermionic(
    objs_ee, objs_eo, objs_oe, objs_oo,
    a_mode: int, b_mode: int
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    Ee_exact, Ee_est, Ee_had = objs_ee
    Eo_exact, Eo_est, Eo_had = objs_eo
    Oe_exact, Oe_est, Oe_had = objs_oe
    Oo_exact, Oo_est, Oo_had = objs_oo

    if a_mode == b_mode:
        coeff = (-0.5j)
        Rho_exact_f = coeff * Ee_exact 
        Rho_est_f   = coeff * Ee_est
        Rho_had_f   = coeff * Ee_had
        return Rho_exact_f, Rho_est_f, Rho_had_f

    def comb(Aee, Aeo, Aoe, Aoo):
        return 0.25 * (Aee + 1j*Aeo - 1j*Aoe + Aoo)

    Rho_exact_f = comb(Ee_exact, Eo_exact, Oe_exact, Oo_exact)
    Rho_est_f   = comb(Ee_est,   Eo_est,   Oe_est,   Oo_est)
    Rho_had_f   = comb(Ee_had,   Eo_had,   Oe_had,   Oo_had)
    return Rho_exact_f, Rho_est_f, Rho_had_f


def simulate_hadamard_test(A_matrix, shots_per_element):
    """
    Simulates the Hadamard test estimation using Binomial sampling
    (which perfectly mimics the sum of independent Bernoulli trials).
    """
    # 1. Calculate the Bernoulli probability of measuring the |0> state (maps to +1)
    prob_0 = (1 + A_matrix) / 2
    
    # 2. Sample the successes directly using the Binomial distribution.
    # This acts exactly like running 'shots_per_element' Bernoulli trials and counting the 1s.
    # The output shape is just 2D: (rows, cols), saving massive amounts of memory.
    successes = np.random.binomial(n=shots_per_element, p=prob_0)
    
    # 3. Map the Bernoulli successes back to the quantum expectation value.
    # Total sum of outcomes = (+1 * successes) + (-1 * failures)
    # failures = shots_per_element - successes
    # A_estimated = total sum / shots_per_element
    A_estimated = (2.0 * successes - shots_per_element) / shots_per_element
    
    return A_estimated
    
def compute_Rho_combo_for_majorana_pair(
    psi_uniform: np.ndarray,
    gamma_map,
    n_modes: int,
    n_qubits: int,
    K: np.ndarray,
    t_list: List[float],
    theta_list: List[float],
    shots_est: int,
    shots_had: int,
    alpha: int,
    beta: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    R_exact = []
    R_est = []
    R_had = []
    C_Exact = []
    
    for t_val, theta_val in zip(t_list, theta_list):
        O = conj_exp_majorana_bilinear_matrix_quadratic(
            gamma_map, n_modes=n_modes, n_qubits=n_qubits,
            K=K, t=t_val, theta=theta_val, a=alpha, b=beta
        )
        psi_TT = O @ psi_uniform.copy()
        psi_TT /= np.linalg.norm(psi_TT)

        C_exact = correlation_matrix_majoranas_heisenberg(psi_TT, gamma_map, n_modes, n_qubits)
        C_Exact.append(C_exact)
        rho_exact = estimate_fermionic_1body_observable(C_exact, n_modes)
        R_exact.append(rho_exact)
        if theta_val == theta_list[0]:
            shots_est_good = int(0.5*shots_est)
        else:
            shots_est_good = int(0.25*shots_est)
        C_est = correlation_matrix_majoranas_from_statevector_via_shadows(
            psi_TT, gamma_map=gamma_map, n_modes=n_modes, n_qubits=n_qubits, shots=shots_est_good
        )
        rho_est = estimate_fermionic_1body_observable(C_est, n_modes)
        R_est.append(rho_est)


    Rho_exact_pair = -1j * (2*R_exact[0] - R_exact[1] - R_exact[2])
    Rho_est_pair   = -1j * (2*R_est[0]   - R_est[1]   - R_est[2])
    C_exact_pair   = -1j * (2*C_Exact[0]   - C_Exact[1]   - C_Exact[2])
    C_exact_pair *= 1/(2j)
    C_exact_pair = np.real(C_exact_pair)
    C_had = simulate_hadamard_test(C_exact_pair, shots_had)
    C_had = C_had * 2j
    Rho_had_pair   = estimate_fermionic_commutator_observable(C_had, n_modes)
    
    return Rho_exact_pair, Rho_est_pair, Rho_had_pair



In [16]:
# ============================================================
# Example: compare exact vs estimated
# ============================================================

if __name__ == '__main__':
    n_modes = 8
    t = 1
    a = 1
    b = 2
    shots_budget = 80000

    gamma_jw = jw_gamma_to_pauli(n_modes)
    gamma_tt, n_qubits_tt = ternary_tree_majorana_map(n_modes)

    psi_vac_jw = np.zeros(1 << n_modes, dtype=np.complex128)
    psi_vac_jw[0] = 1.0

    psi_vac_tt = build_tt_fermionic_vacuum_state(
        gamma_tt, n_modes=n_modes, n_qubits_tt=n_qubits_tt,
        check_commutation=True, verbose=False
    )

    psi_uniform_jw = build_single_particle_uniform_state(psi_vac_jw, gamma_jw, n_modes)
    psi_uniform_tt = build_single_particle_uniform_state(psi_vac_tt, gamma_tt, n_modes)

    Vnn, delta = 1, 0.4
    mu = select_mu_for_uniform_manybody_ground_state(n_modes, Vnn)
    h_ring = h_eq14_1d_nn(n_modes, Vnn, delta, mu, periodic=True)
    K = h_to_majorana_K(h_ring)

    twoN = 2 * n_modes

    # Pre-compute Majorana matrices ONCE
    G_list = []
    for c in range(twoN):
        Pk, coeff = paulisum_to_single_term(gamma_tt(c))
        G_list.append(coeff * pauli_key_to_matrix(Pk, n_qubits_tt))

    t_list     = [-t, -t, -t]
    theta_list = [np.pi/4, 0.0, np.pi/2]

    a_mode = a
    b_mode = b

    n_theta = len(t_list)

    # fermionic operator uses 1 Majorana bilinear if a=b, else 4 bilinears
    n_majorana_pairs = 1 if (a_mode == b_mode) else 4

    twoN = 2 * n_modes            # number of Majorana operators γ_0,...,γ_{2n-1}
    n_corr_entries = twoN * twoN  # size of the full Majorana correlation matrix C_{ab}

    # ---- classical shadows shot accounting ----
    calls_est = n_majorana_pairs  

    # ---- hadamard/Pauli-measurement shot accounting ----
    calls_had = n_majorana_pairs * 3 * n_corr_entries

    shots_est = np.max([1,int(shots_budget/calls_est)])
    shots_had = np.max([1,int(shots_budget/calls_had)])
    
    num_loops = 10
    total_start_time = time.perf_counter()


    max_errors_shadow = []
    max_errors_had = []

    
    for loop_idx in range(num_loops):
        print(f"\n{'='*50}")
        print(f"ITERATION {loop_idx + 1} / {num_loops}")
        print(f"{'='*50}")

        start_time = time.perf_counter()

        t_used = t_list[0]  # (=-t) match your theta-loop

        if a_mode == b_mode:
            ae, ao = 2*a_mode, 2*a_mode + 1

            objs_ao = compute_Rho_combo_for_majorana_pair(
                psi_uniform=psi_uniform_tt, gamma_map=gamma_tt,
                n_modes=n_modes, n_qubits=n_qubits_tt, K=K,
                t_list=t_list, theta_list=theta_list,
                shots_est=shots_est, shots_had=shots_had,
                alpha=ae, beta=ao
            )

            Rho_exact_f, Rho_est_f, Rho_had_loop_f = combine_majorana_pair_objects_to_fermionic(
                objs_ao, objs_ao, objs_ao, objs_ao, a_mode=a_mode, b_mode=b_mode
            )

        else:
            ae, ao = 2*a_mode, 2*a_mode + 1
            be, bo = 2*b_mode, 2*b_mode + 1

            objs_ee = compute_Rho_combo_for_majorana_pair(
                psi_uniform_tt, gamma_tt, n_modes, n_qubits_tt, K,
                t_list, theta_list, shots_est, shots_had, ae, be
            )
            objs_eo = compute_Rho_combo_for_majorana_pair(
                psi_uniform_tt, gamma_tt, n_modes, n_qubits_tt, K,
                t_list, theta_list, shots_est, shots_had, ae, bo
            )
            objs_oe = compute_Rho_combo_for_majorana_pair(
                psi_uniform_tt, gamma_tt, n_modes, n_qubits_tt, K,
                t_list, theta_list, shots_est, shots_had, ao, be
            )
            objs_oo = compute_Rho_combo_for_majorana_pair(
                psi_uniform_tt, gamma_tt, n_modes, n_qubits_tt, K,
                t_list, theta_list, shots_est, shots_had, ao, bo
            )

            Rho_exact_f, Rho_est_f, Rho_had_loop_f = combine_majorana_pair_objects_to_fermionic(
                objs_ee, objs_eo, objs_oe, objs_oo, a_mode=a_mode, b_mode=b_mode
            )
        max_errors_shadow.append(np.max(np.abs(Rho_exact_f - Rho_est_f)))
        max_errors_had.append(np.max(np.abs(Rho_exact_f - Rho_had_loop_f)))
        
        print("\nConsistency checks:")
        print(f"|Rho_exact_f - Rho_est_f|:            {np.max(np.abs(Rho_exact_f - Rho_est_f)):.6e}")
        print(f"|Rho_exact_f - Rho_had_f|:       {np.max(np.abs(Rho_exact_f - Rho_had_loop_f)):.6e}")
        end_time = time.perf_counter()
        print(f"\nIteration {loop_idx + 1} elapsed time: {end_time - start_time:.6f} seconds")

    total_end_time = time.perf_counter()
    print(f"\n{'='*50}")
    print(n_modes)
    print(np.mean(max_errors_shadow))
    print(np.mean(max_errors_had))
    print(np.std(max_errors_shadow))
    print(np.std(max_errors_had))
    print(f"ALL 10 ITERATIONS COMPLETED IN: {total_end_time - total_start_time:.6f} seconds")
    print(f"{'='*50}")


ITERATION 1 / 10

Consistency checks:
|Rho_exact_f - Rho_est_f|:            1.942874e-01
|Rho_exact_f - Rho_had_f|:       2.285909e-01

Iteration 1 elapsed time: 49.035136 seconds

ITERATION 2 / 10

Consistency checks:
|Rho_exact_f - Rho_est_f|:            1.869241e-01
|Rho_exact_f - Rho_had_f|:       1.971233e-01

Iteration 2 elapsed time: 49.204506 seconds

ITERATION 3 / 10

Consistency checks:
|Rho_exact_f - Rho_est_f|:            2.539143e-01
|Rho_exact_f - Rho_had_f|:       1.961807e-01

Iteration 3 elapsed time: 48.852203 seconds

ITERATION 4 / 10

Consistency checks:
|Rho_exact_f - Rho_est_f|:            1.915451e-01
|Rho_exact_f - Rho_had_f|:       1.930098e-01

Iteration 4 elapsed time: 48.853404 seconds

ITERATION 5 / 10

Consistency checks:
|Rho_exact_f - Rho_est_f|:            2.175369e-01
|Rho_exact_f - Rho_had_f|:       2.617522e-01

Iteration 5 elapsed time: 48.813984 seconds

ITERATION 6 / 10

Consistency checks:
|Rho_exact_f - Rho_est_f|:            1.804762e-01
|Rho_